In [39]:
from abc import ABC
from abc import abstractmethod
from argparse import ArgumentParser
from argparse import Namespace
from collections import defaultdict
import datetime
from functools import reduce
import hashlib
import itertools
import multiprocessing
from multiprocessing import Queue
import os
import random
import sys
import time
import traceback
from types import SimpleNamespace
from typing import Any, Callable, Dict, List, Tuple, Type

import pandas as pd
import yaml

from brevitas_examples.common.benchmark.utils import BenchmarkUtils
from brevitas_examples.llm.benchmark.llm_benchmark import LLMBenchmarkUtils

def parse_results(entrypoint_utils: BenchmarkUtils, results_folder: str) -> pd.DataFrame:
    row_data_list = []
    job_config = None
    for entry in os.scandir(results_folder):
        if entry.is_dir() and entry.name not in ["__pycache__"]:
            # Get the identifier of the job
            job_name = os.path.basename(entry.path)
            # Retrieve the configuration from the YAML file
            with open(f"{results_folder}/{job_name}/config.yaml", 'r') as f:
                job_config = yaml.safe_load(f)
            try:
                with open(f"{results_folder}/{job_name}/run_results.yaml", 'r') as f:
                    job_results = yaml.safe_load(f)
            except Exception:
                # Failsafe if entrypoint failed in a way that brings down the whole process
                job_results = {
                    "status": "crashed",
                    "elapsed_time": -1.,
                    "retry_number": -1.,
                    "brevitas_version": -1.,
                    "torch_version": -1.,}
            # If the job was not succesful, try parsing the log
            if job_results["status"] == "crashed":
                # Load the log file
                with open(f"{results_folder}/{job_name}/stdout.out", 'r') as f:
                    job_log = f.read()
                    # Parse results from log
                    job_log_results = entrypoint_utils.parse_log(job_log)
                # Manually populate the results
                job_results = {
                    "elapsed_time": job_results["elapsed_time"],
                    "status": job_results["status"],
                    "retry_number": job_results["retry_number"],
                    "brevitas_version": job_results["brevitas_version"],
                    "torch_version": job_results["torch_version"],
                    **job_log_results,}
            # Add entry to DataFrame
            row_data = {"job_id": job_name, **job_config, **job_results}
            row_data_list.append(row_data)
    if job_config is not None:
        # Columns are obtained by computing the union of the sets of keys in row_data_list, since,
        # for instance, some jobs might have crashed before completing the LM eval
        common_keys = ["job_id"] + list(job_config.keys()) + [
            "elapsed_time", "status", "retry_number", "brevitas_version", "torch_version"
        ] + entrypoint_utils.eval_metrics
        columns = common_keys
        #common_keys_set = set(common_keys)
        #columns = common_keys + list(
        #    reduce(lambda x, y: x.union(y), [set(row_data.keys()) for row_data in row_data_list
        #                                    ]).difference(common_keys_set))
        # Instantiate DataFrame to store the results
        df = pd.DataFrame(columns=columns)
        print(len(row_data_list))
        for row_data in row_data_list:
            # Fill missing columns with None
            df.loc[len(df)] = [row_data[key] if key in row_data else None for key in columns]
    else:
        raise ValueError(f"No experiments results were found in {results_folder}")
    return df, row_data_list

entrypoint_utils = LLMBenchmarkUtils
results_folder = "/home/pmonteag/clones/brevitas/src/brevitas_examples/llm/benchmark/results"

df, row_data_list = parse_results(entrypoint_utils=entrypoint_utils, results_folder=results_folder)


27


/tmp/ipykernel_3518916/1830472180.py:81: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df.loc[len(df)] = [row_data[key] if key in row_data else None for key in columns]
/tmp/ipykernel_3518916/1830472180.py:81: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df.loc[len(df)] = [row_data[key] if key in row_data else None for key in columns]
/tmp/ipykernel_3518916/1830472180.py:81: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, t

In [19]:
import pandas as pd  

# df = df[(df['quant_ppl'] < 1000.)]
#df = df[(df['learning_rate'] == 0.1)]
# Select certain columns  
df = pd.read_csv('/home/pmonteag/clones/brevitas/src/brevitas_examples/llm/benchmark/new_results_scales_fix/results.csv')  
selected_columns = ['job_id', 'model', 'weight_bit_width', 'weight_quant_granularity', 'learned_round_lr', 'learned_round_scale', 'learned_round_iters', 'quant_ppl', 'all_acc']  # Change these to your actual column names  
df = df[selected_columns]

CONFIGS = {
    "baseline": (0.005, False, 200),
    "baseline_lr": (0.001, False, 200),
    "best": (0.001, False, 1000),
    "best_lr": (0.005, False, 1000),
    "scale": (0.005, True, 200),
    "scale_best": (0.001, True, 1000),
    "scale_best_lr": (0.005, True, 1000),
}
curr_config = "scale"

df = df[
    (df['learned_round_lr'] == CONFIGS[curr_config][0]) & 
    (df['learned_round_scale'] == CONFIGS[curr_config][1]) & 
    (df['learned_round_iters'] == CONFIGS[curr_config][2])
]

df['model'] = df['model'].apply(lambda x : {'Qwen/Qwen2.5-3B-Instruct': 'Qwen2.5-3B-Instruct', 'facebook/opt-125m': 'opt-125m', 'meta-llama/Llama-3.2-1B-Instruct': 'Llama-3.2-1B-Instruct'}[x])  
# Display the filtered DataFrame  
df = df.sort_values(['model','weight_bit_width', 'weight_quant_granularity'])
print(df)  
print(len(df))

df.to_csv(f'auto_round_{curr_config}.csv')

                              job_id                  model  weight_bit_width  \
20  f68f11d5a20cfced42e47ddf740d81f0  Llama-3.2-1B-Instruct                 2   
1   dc1763bde1dcb729317c1987081bfc05  Llama-3.2-1B-Instruct                 4   
19  6d87df3a7960d03f5efe6e99e5e99b9c  Llama-3.2-1B-Instruct                 4   
29  034f15f07608c579a67dc648323b81d5    Qwen2.5-3B-Instruct                 2   
12  8577fa7d49fe8badc3c9d94c38ef413f    Qwen2.5-3B-Instruct                 4   
16  f6bcea96155b0748a5a221401e24cc83    Qwen2.5-3B-Instruct                 4   
18  694d15a26b0ac66b4f4caf57e9b57738               opt-125m                 2   
9   13339a0a150143f7eb2ce04387838695               opt-125m                 4   
10  ed6459cfd4b2db162976fc504da1025f               opt-125m                 4   

   weight_quant_granularity  learned_round_lr  learned_round_scale  \
20                per_group             0.005                 True   
1               per_channel             0.005    

In [8]:
import pandas as pd  

# df = df[(df['quant_ppl'] < 1000.)]
#df = df[(df['learning_rate'] == 0.1)]
# Select certain columns  
df = pd.read_csv('/home/pmonteag/clones/brevitas/src/brevitas_examples/llm/benchmark/new_results_scales_fix/results.csv')  
selected_columns = ['job_id', 'model', 'weight_bit_width', 'weight_quant_granularity', 'learned_round_lr', 'learned_round_scale', 'learned_round_iters', 'quant_ppl', 'all_acc']  # Change these to your actual column names  
df = df[selected_columns]

CONFIGS = {
    "baseline": (0.005, False, 200),
    "baseline_lr": (0.001, False, 200),
    "best": (0.001, False, 1000),
    "best_lr": (0.005, False, 1000),
    "scale": (0.005, True, 200),
    "scale_best": (0.001, True, 1000),
    "scale_best_lr": (0.005, True, 1000),
}
curr_config = "scale"

df = df[
    (df['learned_round_lr'] == CONFIGS[curr_config][0]) & 
    (df['learned_round_scale'] == CONFIGS[curr_config][1]) & 
    (df['learned_round_iters'] == CONFIGS[curr_config][2])
]

df['model'] = df['model'].apply(lambda x : {'Qwen/Qwen2.5-3B-Instruct': 'Qwen2.5-3B-Instruct', 'facebook/opt-125m': 'opt-125m', 'meta-llama/Llama-3.2-1B-Instruct': 'Llama-3.2-1B-Instruct'}[x])  
# Display the filtered DataFrame  
df = df.sort_values(['model','weight_bit_width', 'weight_quant_granularity'])
print(df)  
print(len(df))

df.to_csv(f'auto_round_fix_{curr_config}.csv')

                              job_id                  model  weight_bit_width  \
20  f68f11d5a20cfced42e47ddf740d81f0  Llama-3.2-1B-Instruct                 2   
1   dc1763bde1dcb729317c1987081bfc05  Llama-3.2-1B-Instruct                 4   
19  6d87df3a7960d03f5efe6e99e5e99b9c  Llama-3.2-1B-Instruct                 4   
29  034f15f07608c579a67dc648323b81d5    Qwen2.5-3B-Instruct                 2   
12  8577fa7d49fe8badc3c9d94c38ef413f    Qwen2.5-3B-Instruct                 4   
16  f6bcea96155b0748a5a221401e24cc83    Qwen2.5-3B-Instruct                 4   
18  694d15a26b0ac66b4f4caf57e9b57738               opt-125m                 2   
9   13339a0a150143f7eb2ce04387838695               opt-125m                 4   
10  ed6459cfd4b2db162976fc504da1025f               opt-125m                 4   

   weight_quant_granularity  learned_round_lr  learned_round_scale  \
20                per_group             0.005                 True   
1               per_channel             0.005    

In [1]:
import pandas as pd  

# df = df[(df['quant_ppl'] < 1000.)]
#df = df[(df['learning_rate'] == 0.1)]
# Select certain columns  
df = pd.read_csv('/home/pmonteag/clones/brevitas/src/brevitas_examples/llm/benchmark/new_results_grid/results.csv')  
selected_columns = ['model', 'weight_bit_width', 'weight_quant_granularity', 'learned_round_scale_lr', 'learned_round_scale_momentum', 'learned_round_scale', 'learned_round_iters', 'quant_ppl', 'all_acc']  # Change these to your actual column names  
df = df[selected_columns]

CONFIGS = {
    "baseline": (0.005, False, 200),
    "baseline_lr": (0.001, False, 200),
    "best": (0.001, False, 1000),
    "best_lr": (0.005, False, 1000),
    "scale": (0.005, True, 200),
    "scale_best": (0.001, True, 1000),
    "scale_best_lr": (0.005, True, 1000),
}
curr_config = "qwen"

#df = df[
#    (df['learned_round_lr'] == CONFIGS[curr_config][0]) & 
#    (df['learned_round_scale'] == CONFIGS[curr_config][1]) & 
#    (df['learned_round_iters'] == CONFIGS[curr_config][2])
#]

df['model'] = df['model'].apply(lambda x : {'Qwen/Qwen2.5-3B-Instruct': 'Qwen2.5-3B-Instruct', 'facebook/opt-125m': 'opt-125m', 'meta-llama/Llama-3.2-1B-Instruct': 'Llama-3.2-1B-Instruct'}[x])  
# df = df[(df['model'] == 'Qwen2.5-3B-Instruct') & (df['weight_bit_width'] == '2')]
# Display the filtered DataFrame  
df = df.sort_values(['model','weight_bit_width', 'weight_quant_granularity'])
#print(df)  
#print(len(df))

# Group by 'group' and 'subgroup' and get indices of rows with maximum 'metric'  
idx = df.groupby(['model','weight_bit_width', 'weight_quant_granularity'])['all_acc'].idxmax()  

df = df.loc[idx]

print(df)

df.to_csv(f'auto_round_{curr_config}.csv')

                    model  weight_bit_width weight_quant_granularity  \
11  Llama-3.2-1B-Instruct                 2                per_group   
27  Llama-3.2-1B-Instruct                 4              per_channel   
29  Llama-3.2-1B-Instruct                 4                per_group   
0     Qwen2.5-3B-Instruct                 2                per_group   
44    Qwen2.5-3B-Instruct                 4              per_channel   
19    Qwen2.5-3B-Instruct                 4                per_group   

    learned_round_scale_lr  learned_round_scale_momentum  learned_round_scale  \
11                   0.010                           0.1                 True   
27                   0.005                           0.9                 True   
29                   0.005                           0.9                 True   
0                    0.005                           0.0                 True   
44                   0.001                           0.1                 True   
19       

In [1]:
import pandas as pd  

# df = df[(df['quant_ppl'] < 1000.)]
#df = df[(df['learning_rate'] == 0.1)]
# Select certain columns  
df = pd.read_csv('/home/pmonteag/clones/brevitas/src/brevitas_examples/llm/benchmark/new_results_grid_increased/results.csv')  
selected_columns = ['model', 'weight_bit_width', 'weight_quant_granularity', 'learned_round_lr', 'learned_round_scale_lr', 'learned_round_scale_momentum', 'learned_round_scale', 'learned_round_iters', 'quant_ppl', 'all_acc']  # Change these to your actual column names  
df = df[selected_columns]

CONFIGS = {
    "baseline": (0.005, False, 200),
    "baseline_lr": (0.001, False, 200),
    "best": (0.001, False, 1000),
    "best_lr": (0.005, False, 1000),
    "scale": (0.005, True, 200),
    "scale_best": (0.001, True, 1000),
    "scale_best_lr": (0.005, True, 1000),
}
curr_config = "qwen_new"

#df = df[
#    (df['learned_round_lr'] == CONFIGS[curr_config][0]) & 
#    (df['learned_round_scale'] == CONFIGS[curr_config][1]) & 
#    (df['learned_round_iters'] == CONFIGS[curr_config][2])
#]

df['model'] = df['model'].apply(lambda x : {'Qwen/Qwen2.5-3B-Instruct': 'Qwen2.5-3B-Instruct', 'facebook/opt-125m': 'opt-125m', 'meta-llama/Llama-3.2-1B-Instruct': 'Llama-3.2-1B-Instruct'}[x])  
# df = df[(df['model'] == 'Qwen2.5-3B-Instruct') & (df['weight_bit_width'] == '2')]
# Display the filtered DataFrame  
df = df.sort_values(['model','weight_bit_width', 'weight_quant_granularity'])
#print(df)  
#print(len(df))

# Group by 'group' and 'subgroup' and get indices of rows with maximum 'metric'  
idx = df.groupby(['model','weight_bit_width', 'weight_quant_granularity'])['all_acc'].idxmax()  

df = df.loc[idx]

print(df)

df.to_csv(f'auto_round_{curr_config}.csv')

                  model  weight_bit_width weight_quant_granularity  \
10  Qwen2.5-3B-Instruct                 2                per_group   
26  Qwen2.5-3B-Instruct                 4              per_channel   
2   Qwen2.5-3B-Instruct                 4                per_group   

    learned_round_lr  learned_round_scale_lr  learned_round_scale_momentum  \
10             0.005                   0.001                           0.0   
26             0.005                   0.001                           0.1   
2              0.002                   0.001                           0.0   

    learned_round_scale  learned_round_iters  quant_ppl   all_acc  
10                 True                  500  20.859011  0.523103  
26                 True                  500   8.797758  0.638278  
2                  True                  500   8.239154  0.646126  


In [6]:
import pandas as pd  

# df = df[(df['quant_ppl'] < 1000.)]
#df = df[(df['learning_rate'] == 0.1)]
# Select certain columns  
df = pd.read_csv('/home/pmonteag/clones/brevitas/src/brevitas_examples/llm/benchmark/results_act_quant_spinquant/results.csv')  
selected_columns = ['model', 'weight_bit_width', 'weight_quant_granularity', 'learned_round_lr', 'learned_round_scale_lr', 'learned_round_scale_momentum', 'learned_round_scale', 'learned_round_iters', 'quant_ppl', 'all_acc', 'leaderboard:arc:challenge:0_acc', 'lighteval:arc:easy:0_acc', 'leaderboard:hellaswag:0_acc', 'leaderboard:winogrande:0_acc', 'lighteval:piqa:0_acc']  # Change these to your actual column names  
df = df[selected_columns]

CONFIGS = {
    "baseline": (0.005, False, 200),
    "baseline_lr": (0.001, False, 200),
    "best": (0.001, False, 1000),
    "best_lr": (0.005, False, 1000),
    "scale": (0.005, True, 200),
    "scale_best": (0.001, True, 1000),
    "scale_best_lr": (0.005, True, 1000),
}
curr_config = "qwen_new"

#df = df[
#    (df['learned_round_lr'] == CONFIGS[curr_config][0]) & 
#    (df['learned_round_scale'] == CONFIGS[curr_config][1]) & 
#    (df['learned_round_iters'] == CONFIGS[curr_config][2])
#]

df['model'] = df['model'].apply(lambda x : {'Qwen/Qwen2.5-3B-Instruct': 'Qwen2.5-3B-Instruct', 'facebook/opt-125m': 'opt-125m', 'meta-llama/Llama-3.2-1B-Instruct': 'Llama-3.2-1B-Instruct'}[x])  
df = df[(df['model'] == 'Qwen2.5-3B-Instruct') | (df['model'] == 'Llama-3.2-1B-Instruct')]
# Display the filtered DataFrame  
df = df.sort_values(['model','weight_bit_width', 'weight_quant_granularity'])
#print(df)  
#print(len(df))

# Group by 'group' and 'subgroup' and get indices of rows with maximum 'metric'  
idx = df.groupby(['model','weight_bit_width', 'weight_quant_granularity'])['all_acc'].idxmax()  
df = df.loc[idx]

print(df)

df.to_csv(f'auto_round_{curr_config}.csv')

                    model  weight_bit_width weight_quant_granularity  \
20  Llama-3.2-1B-Instruct                 4              per_channel   
5     Qwen2.5-3B-Instruct                 4              per_channel   

    learned_round_lr  learned_round_scale_lr  learned_round_scale_momentum  \
20             0.002                   0.001                           0.9   
5              0.005                   0.001                           0.1   

    learned_round_scale  learned_round_iters  quant_ppl   all_acc  \
20                 True                  500  17.767565  0.501492   
5                  True                  500  10.505976  0.577798   

    leaderboard:arc:challenge:0_acc  lighteval:arc:easy:0_acc  \
20                         0.302901                  0.593434   
5                          0.407850                  0.696128   

    leaderboard:hellaswag:0_acc  leaderboard:winogrande:0_acc  \
20                     0.395340                      0.542226   
5             